In [1]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")

checkpoint_run_dir = (
    drive_root
    / "checkpoints"
    / "vit5_finetuning_20261106_161925_175323"
)

if not (checkpoint_run_dir / "best").is_dir():
    raise FileNotFoundError(
        "Lỗi: Không tìm thấy thư mục checkpoint best: "
        + str(checkpoint_run_dir / "best")
    )

print("Checkpoint:", checkpoint_run_dir / "best")

Mounted at /content/drive
Checkpoint: /content/drive/MyDrive/vietnamese-summarization/checkpoints/vit5_finetuning_20261106_161925_175323/best


In [2]:
import os
import sys
import subprocess
from pathlib import Path

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = "c38254699f346b9c1205cb2b3d567fa04fbbe174"

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True,
    )

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository không đúng dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError(
        "Lỗi: Repository còn thay đổi. Hãy dùng runtime mới."
    )

subprocess.run(
    ["git", "fetch", "origin"],
    cwd=repo_dir,
    check=True,
)

subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Thư mục mã nguồn:", repo_dir)
print("Commit nền:", target_commit)

Thư mục mã nguồn: /content/vietnamese-summarization
Commit nền: c38254699f346b9c1205cb2b3d567fa04fbbe174


In [3]:
%pip uninstall -y diffusers torchvision timm fastai
%pip install "jedi==0.20.1"
%pip install -r /content/vietnamese-summarization/requirements.txt
%pip check

Found existing installation: diffusers 0.42.1
Uninstalling diffusers-0.42.1:
  Successfully uninstalled diffusers-0.42.1
Found existing installation: torchvision 0.29.1+cu130
Uninstalling torchvision-0.29.1+cu130:
  Successfully uninstalled torchvision-0.29.1+cu130
Found existing installation: timm 1.0.31
Uninstalling timm-1.0.31:
  Successfully uninstalled timm-1.0.31
Found existing installation: fastai 2.8.12
Uninstalling fastai-2.8.12:
  Successfully uninstalled fastai-2.8.12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 15.2 MB/s  0:00:00
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 72.1 MB/s  0:00:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 207.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 7

In [4]:
import json
import torch
from importlib.metadata import version
from src.utils import load_generation_config, load_train_config

if not torch.cuda.is_available():
    raise RuntimeError("Lỗi: GPU chưa khả dụng để chạy demo.")

train_config = load_train_config(repo_dir / "configs" / "train.json")
generation_config = load_generation_config(
    repo_dir / "configs" / "generation.json"
)

print(json.dumps(
    {
        "pytorch_version": str(torch.__version__),
        "transformers_version": version("transformers"),
        "gradio_version": version("gradio"),
        "gpu_name": torch.cuda.get_device_name(0),
        "max_source_length": train_config["max_source_length"],
        "max_target_length": train_config["max_target_length"],
        "generation_config": generation_config,
    },
    ensure_ascii=False,
    indent=2,
))

{
  "pytorch_version": "2.11.0+cu130",
  "transformers_version": "4.57.6",
  "gradio_version": "6.17.3",
  "gpu_name": "Tesla T4",
  "max_source_length": 1024,
  "max_target_length": 128,
  "generation_config": {
    "num_beams": 4,
    "max_new_tokens": 128,
    "do_sample": false
  }
}


In [5]:
import app.gradio_app

print("Đã import giao diện từ:", app.gradio_app.__file__)

Đã import giao diện từ: /content/vietnamese-summarization/app/gradio_app.py


In [6]:
if "demo_summarizer" not in globals():
    demo_summarizer = app.gradio_app.DemoSummarizer(
        checkpoint_run_dir=checkpoint_run_dir,
        config_path=repo_dir / "configs" / "train.json",
        generation_config_path=repo_dir / "configs" / "generation.json",
    )
    print("Đã tải model và tokenizer từ checkpoint best.")
else:
    print("Dùng lại model và tokenizer đã tải.")

print("Checkpoint:", checkpoint_run_dir / "best")
print("Thiết bị:", next(demo_summarizer.model.parameters()).device)
print("Thông tin chọn checkpoint:", demo_summarizer.selection)

Đã tải model và tokenizer từ checkpoint best.
Checkpoint: /content/drive/MyDrive/vietnamese-summarization/checkpoints/vit5_finetuning_20261106_161925_175323/best
Thiết bị: cuda:0
Thông tin chọn checkpoint: {'metric': 'validation_loss', 'mode': 'min', 'validation_loss': 2.110764804105354, 'epoch': 2, 'global_step': 500, 'validation_sample_ids': ['vi_0247f2933f242fed13763b11d7c0374011457c56addec08b7df3a279b33e7af9', 'vi_06dfbd1da77e8b104d493a9ac025815a362666ae82e0fcfed48bb34994ce2f4d', 'vi_07e6cf42893b328cb65aaecf12dd4591fda7fc3fa592952bc157263596518736', 'vi_090014818463c626f8dadb527bec4abbe0a6f6134fb2ac7c0bf2e57d661bbc67', 'vi_0941e0da08213a4e664f319723653a0bf9d70eb25c75c45cd635f68c92118b13', 'vi_0c43b2e82368b2d3966b92d3b4895514494cabb1086bbbe44a842ea9be9648e4', 'vi_0d7fe72400e57e4d786f88d3bf41525eaa9895872976233ae2ea7907524d2c78', 'vi_131d6fd921cd68eeb99170a69c1431504cd5e7f1c0b08e6280678b3cfb0eea9c', 'vi_137c9324608f0765e0ae3751d68232305decf0860cd9f3832b53a5cac05c7fab', 'vi_138393089f

In [7]:
if "demo_summarizer" not in globals():
    raise RuntimeError("Lỗi: Chưa khởi tạo model cho giao diện.")

if "demo" in globals():
    demo.close()

demo = app.gradio_app.create_demo(demo_summarizer)

demo.launch(
    share=True,
    inline=True,
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed963654296fe07de4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
import json

short_text = (
    "Rửa sạch hạt đậu xanh, ngâm khoảng 8 giờ rồi gieo vào chậu đất ẩm. "
    "Đặt chậu nơi có ánh sáng và tưới nước vừa đủ."
)

long_text = short_text * 300

test_cases = [
    {"name": "empty", "text": "", "expected": "empty"},
    {"name": "whitespace", "text": " \n\t ", "expected": "empty"},
    {"name": "short", "text": short_text, "expected": "normal"},
    {"name": "long", "text": long_text, "expected": "truncated"},
    {"name": "short_after_long", "text": short_text, "expected": "normal"},
]

model_before = demo_summarizer.model
tokenizer_before = demo_summarizer.tokenizer
results = []
all_passed = True

for test_case in test_cases:
    summary, message = demo_summarizer.summarize(test_case["text"])

    if test_case["expected"] == "empty":
        passed = summary == "" and message != ""
    elif test_case["expected"] == "truncated":
        passed = summary != "" and message != ""
    else:
        passed = summary != "" and message == ""

    if not passed:
        all_passed = False

    results.append({
        "name": test_case["name"],
        "summary": summary,
        "message": message,
        "passed": passed,
    })

model_reused = demo_summarizer.model is model_before
tokenizer_reused = demo_summarizer.tokenizer is tokenizer_before

report = {
    "cases": results,
    "model_reused": model_reused,
    "tokenizer_reused": tokenizer_reused,
    "all_passed": all_passed and model_reused and tokenizer_reused,
}

print(json.dumps(report, ensure_ascii=False, indent=2))

if not report["all_passed"]:
    raise RuntimeError("Lỗi: Có trường hợp kiểm tra demo chưa đạt.")

{
  "cases": [
    {
      "name": "empty",
      "summary": "",
      "message": "Vui lòng nhập văn bản cần tóm tắt.",
      "passed": true
    },
    {
      "name": "whitespace",
      "summary": "",
      "message": "Vui lòng nhập văn bản cần tóm tắt.",
      "passed": true
    },
    {
      "name": "short",
      "summary": "Rửa sạch hạt đậu xanh và hạt đậu xanh.",
      "message": "",
      "passed": true
    },
    {
      "name": "long",
      "summary": "Đặt chậu nơi có ánh sáng và tưới nước vừa đủ. Đặt chậu nơi có ánh sáng và tưới nước vừa đủ. Đặt chậu nơi có ánh sáng và tưới nước vừa đủ. Đặt chậu nơi có ánh sáng và tưới nước vừa đủ. Đặt chậu nơi có ánh sáng và tưới nước vừa đủ. Đặt chậu nơi có ánh sáng và tưới nước vừa đủ. Đặt chậu nơi có ánh sáng và tưới nước vừa đủ. Đặt chậu nơi có ánh sáng và tưới nước vừa đủ. Đặt chậu nơi có ánh sáng và tưới nước vừa đủ. Đặt chậu nơi có ánh sáng và tưới nước vừa đủ. Đặt chậu nơi có ánh sáng và tưới",
      "message": "Văn bản có 9000 to

In [9]:
import json
import platform
import subprocess
from datetime import datetime, timezone
from importlib.metadata import version
from google.colab import files
from src.utils import load_generation_config, load_train_config

commit_result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError("Lỗi: Mã nguồn đã thay đổi trong runtime.")

if commit_result.stdout.strip() != target_commit:
    raise RuntimeError("Lỗi: Runtime không dùng đúng commit đã chọn.")

train_config = load_train_config(repo_dir / "configs" / "train.json")
generation_config = load_generation_config(
    repo_dir / "configs" / "generation.json"
)

created_at = datetime.now(timezone.utc)
fallback_id = "demo_fallback_" + created_at.strftime("%Y%m%d_%H%M%S_%f")

metadata = {
    "purpose": "presentation_fallback",
    "input_origin": "Ba đầu vào tự soạn; chưa đối chiếu trùng với tập train.",
    "created_at_utc": created_at.isoformat(),
    "git_commit": commit_result.stdout.strip(),
    "checkpoint_run_id": checkpoint_run_dir.name,
    "checkpoint_directory": str(checkpoint_run_dir / "best"),
    "checkpoint_epoch": demo_summarizer.selection["epoch"],
    "checkpoint_global_step": demo_summarizer.selection["global_step"],
    "checkpoint_validation_loss": demo_summarizer.selection["validation_loss"],
    "model_id": train_config["model"]["id"],
    "model_revision": train_config["model"]["revision"],
    "max_source_length": train_config["max_source_length"],
    "max_target_length": train_config["max_target_length"],
    "generation_config": generation_config,
    "python_version": platform.python_version(),
    "pytorch_version": str(torch.__version__),
    "transformers_version": version("transformers"),
    "gradio_version": version("gradio"),
    "model_device": str(next(demo_summarizer.model.parameters()).device),
    "limitation": (
        "Đầu ra thật có thể lặp, thiếu ý hoặc sai thông tin; "
        "không phải kết quả đánh giá độc lập."
    ),
}

examples = [
    {
        "example_id": "demo_01",
        "source": (
            "Để trồng đậu xanh trong chậu, hãy chọn hạt nguyên vẹn và rửa sạch. "
            "Ngâm hạt trong nước khoảng 8 giờ. Chuẩn bị chậu có lỗ thoát nước "
            "và cho đất tơi xốp vào. Gieo hạt sâu khoảng 2 cm rồi phủ đất mỏng. "
            "Đặt chậu ở nơi có ánh sáng, tưới nhẹ và giữ đất đủ ẩm. "
            "Tránh tưới quá nhiều vì cây có thể bị úng."
        ),
    },
    {
        "example_id": "demo_02",
        "source": (
            "Để mượn sách tại thư viện trường, trước tiên hãy tìm tên sách "
            "trong danh mục và ghi lại vị trí trên giá. Lấy sách rồi kiểm tra "
            "tình trạng các trang. Mang sách cùng thẻ sinh viên đến quầy "
            "để nhân viên ghi nhận lượt mượn. Kiểm tra ngày phải trả trên phiếu. "
            "Trong thời gian mượn, giữ sách sạch và không viết vào sách. "
            "Trả sách đúng hạn tại quầy thư viện."
        ),
    },
    {
        "example_id": "demo_03",
        "source": (
            "Để chuẩn bị buổi họp nhóm làm bài tập, hãy thống nhất thời gian "
            "và địa điểm với các thành viên. Lập danh sách những vấn đề cần "
            "thảo luận rồi gửi tài liệu trước buổi họp. Khi bắt đầu, cử một "
            "người ghi chép và lần lượt trao đổi từng vấn đề. Cuối buổi, "
            "phân công công việc cụ thể và thống nhất hạn hoàn thành. "
            "Gửi lại biên bản cho cả nhóm để mọi người theo dõi."
        ),
    },
]

records = []

for example in examples:
    summary, message = demo_summarizer.summarize(example["source"])

    record = {
        "example_id": example["example_id"],
        "source": example["source"],
        "summary": summary,
        "message": message,
        "metadata": metadata,
    }
    records.append(record)

    print("Ví dụ:", record["example_id"])
    print("Đầu vào:", record["source"])
    print("Tóm tắt:", summary)
    print("Thông báo:", message)

fallback_dir = drive_root / "outputs" / fallback_id
fallback_dir.mkdir(parents=True, exist_ok=True)
fallback_path = fallback_dir / "demo_examples.jsonl"

with fallback_path.open("w", encoding="utf-8") as output_file:
    for record in records:
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")

saved_records = []

with fallback_path.open(encoding="utf-8") as input_file:
    for line in input_file:
        saved_records.append(json.loads(line))

if saved_records != records:
    raise RuntimeError("Lỗi: File kết quả đọc lại không khớp.")

print("Đã lưu và đọc lại đủ", len(saved_records), "ví dụ:", fallback_path)

files.download(str(fallback_path))

Ví dụ: demo_01
Đầu vào: Để trồng đậu xanh trong chậu, hãy chọn hạt nguyên vẹn và rửa sạch. Ngâm hạt trong nước khoảng 8 giờ. Chuẩn bị chậu có lỗ thoát nước và cho đất tơi xốp vào. Gieo hạt sâu khoảng 2 cm rồi phủ đất mỏng. Đặt chậu ở nơi có ánh sáng, tưới nhẹ và giữ đất đủ ẩm. Tránh tưới quá nhiều vì cây có thể bị úng.
Tóm tắt: Trồng đậu xanh trong chậu. Trồng đậu xanh trong chậu. Trồng đậu xanh trong chậu.
Thông báo: 
Ví dụ: demo_02
Đầu vào: Để mượn sách tại thư viện trường, trước tiên hãy tìm tên sách trong danh mục và ghi lại vị trí trên giá. Lấy sách rồi kiểm tra tình trạng các trang. Mang sách cùng thẻ sinh viên đến quầy để nhân viên ghi nhận lượt mượn. Kiểm tra ngày phải trả trên phiếu. Trong thời gian mượn, giữ sách sạch và không viết vào sách. Trả sách đúng hạn tại quầy thư viện.
Tóm tắt: Tìm sách cần mượn tại thư viện trường. Mượn sách tại thư viện trường. Mượn sách tại thư viện trường.
Thông báo: 
Ví dụ: demo_03
Đầu vào: Để chuẩn bị buổi họp nhóm làm bài tập, hãy thống nhất t

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>